# 📘 Mini-projet Segmentation — LE COURS COMPLET (v1, ligne par ligne)

> **Ce fichier** : la version annotée de `mini_projet_segmentation.ipynb`, avec les
> **7 missions résolues**. Un bloc de cours avant chaque cellule, et **chaque ligne de
> code commentée**. L'original (avec les TODO) reste intact à côté.

## 🎯 Le projet en une phrase

Entraîner un **U-Net** à prédire, pour **chaque pixel** d'une photo, s'il appartient à
une **personne** ou au **fond** — le moteur du fond flou de Zoom et de remove.bg.

## Le pipeline complet

```
images + masques → Dataset PyTorch → U-Net → loss (BCE + Dice)
                        ↓
        boucle d'entraînement → meilleur checkpoint → seuil optimal (val) → éval (test)
```

C'est **le même squelette en 10 étapes** que le S9_P4 du cours — seule différence :
la sortie n'est plus un nombre (régression) mais une **image entière de probabilités**
(une par pixel).

| Étape | Ici |
|---|---|
| données | photos + masques (0 = fond, 1,2,3… = personnes) |
| split | 70 / 15 / 15, reproductible |
| Dataset | charge les images **à la demande** (lazy) |
| modèle | U-Net : encoder → bottleneck → decoder + **skip connections** |
| loss | BCE + **Soft Dice** (différentiable) |
| métriques | Dice + IoU (après seuillage) |
| évaluation | seuil choisi sur la **validation**, score final sur le **test** |

# 0 — Imports

Rien de nouveau sauf deux modules « fichiers » (`zipfile`, `urllib`) et le duo
`TF` / `InterpolationMode` de torchvision qui servira aux transformations d'images.

In [ ]:
import os                    # utilitaires systeme (peu utilise ici)
import random                          # le generateur aleatoire de Python (split + flips)
import zipfile                         # ouvrir et decompresser le .zip du dataset
import urllib.request                  # telecharger un fichier par HTTP
from pathlib import Path               # chemins de fichiers modernes (operateur /)

import numpy as np                     # tableaux numeriques (masques, moyennes)
import matplotlib.pyplot as plt        # tous les affichages
from PIL import Image                  # ouvrir les images (modes RGB / L)

import torch                           # tenseurs + autograd
import torch.nn as nn                  # les couches : Conv2d, BatchNorm2d, ReLU...
from torch.utils.data import Dataset, DataLoader      # servir les donnees par lots
from torchvision.transforms import functional as TF   # resize, flip, to_tensor...
from torchvision.transforms import InterpolationMode  # BILINEAR / NEAREST

# Configuration — les graines et les hyperparamètres

## 🧠 Pourquoi QUATRE lignes de seed ?

Un ordinateur ne tire pas de vrai hasard : il déroule une suite **entièrement déterminée
par un nombre de départ** (la *graine*). Même graine → même suite, à chaque exécution.

Il y a **4 générateurs indépendants** dans ce projet, et fixer l'un ne fixe pas les autres :

| Ligne | Générateur | Agit sur |
|---|---|---|
| `random.seed` | Python | le **split** et le **flip** d'augmentation |
| `np.random.seed` | NumPy | (sécurité) |
| `torch.manual_seed` | PyTorch **CPU** | l'**initialisation des poids** + le `shuffle` des loaders |
| `torch.cuda.manual_seed_all` | PyTorch **GPU** | les opérations sur la carte — générateur **à part** ! |

Sans ça : chaque `Run All` donnerait un split, des poids et des flips différents →
impossible de savoir si un changement a **aidé** ou si c'est le hasard du tirage.

## Hyperparamètres vs paramètres

Les **hyperparamètres** = les réglages que TOI tu choisis. Les **paramètres** = les poids
que le réseau apprend. On regroupe les premiers en haut, en constantes, pour les régler
à un seul endroit.

In [ ]:
SEED = 42                              # la graine : 42 = clin d'oeil geek, n'importe quel nombre marche
random.seed(SEED)                      # fige le generateur de Python
np.random.seed(SEED)                   # fige celui de NumPy
torch.manual_seed(SEED)                # fige celui de PyTorch cote CPU
if torch.cuda.is_available():          # s'il y a un GPU...
    torch.cuda.manual_seed_all(SEED)   # ...son generateur A PART doit aussi etre fige

# le patron standard : GPU s'il y en a un, sinon CPU — le code tourne partout
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# Hyperparametres — modifiables
IMG_SIZE = 128        # toutes les images ramenees a 128x128 (un reseau veut une taille FIXE)
BATCH_SIZE = 8        # 8 images traitees d'un coup (compromis vitesse / memoire)
EPOCHS = 5            # nombre de passages sur tout le dataset
BASE_CHANNELS = 16    # largeur du 1er etage du U-Net (les etages suivants doublent)
LEARNING_RATE = 1e-3  # taille du pas de la descente de gradient (l'hyperparametre n1)

print(f"Device : {DEVICE}")            # verifie ou tu tournes AVANT de lancer quoi que ce soit
if torch.cuda.is_available():
    print(f"GPU    : {torch.cuda.get_device_name(0)}")

# 1 — Télécharger le dataset (une seule fois)

## 🧠 Deux idées dans cette cellule

**1. L'idempotence.** Le `if not DATA_ROOT.exists()` rend la cellule **ré-exécutable sans
dégât** : le 1ᵉʳ run télécharge, tous les suivants sautent. (Souviens-toi du bug
`price / 1e6` exécuté 2 fois — ici le garde-fou empêche exactement ça.)

**2. Le `sorted()` est vital.** `glob` renvoie les fichiers dans un ordre **non garanti**.
Or tout le projet repose sur `image_paths[i]` ↔ `mask_paths[i]` = la même personne.
Trier les **deux** listes aligne les index (les noms concordent : `xxx.png` ↔ `xxx_mask.png`).
Sans ça : le réseau apprendrait le masque d'une personne sur la photo d'une autre —
**et rien ne planterait**.

## ⚠️ Piège vécu : le zip imbriqué

Le zip officiel crée parfois `PennFudanPed/PennFudanPed/` (dossier en double). Résultat :
`glob` ne trouve rien → listes vides → `IndexError` à la Mission 1. Le correctif propre
est `next(DATA_ROOT.rglob("PNGImages"))` qui cherche le dossier **où qu'il soit**
(c'est ce que fait la v3).

In [ ]:
DATA_ROOT = Path("../data/PennFudanPed")     # le dossier attendu apres decompression
ZIP_PATH = Path("../data/PennFudanPed.zip")  # ou poser l'archive telechargee
URL = "https://www.cis.upenn.edu/~jshi/ped_html/PennFudanPed.zip"   # la source officielle

if not DATA_ROOT.exists():                           # garde-fou : ne telecharge qu'UNE fois
    print("Telechargement du dataset (~50 Mo)...")
    urllib.request.urlretrieve(URL, ZIP_PATH)        # telechargement HTTP -> fichier local
    with zipfile.ZipFile(ZIP_PATH, "r") as archive:  # ouvre le zip (referme auto par le with)
        archive.extractall(".")                      # decompresse dans le dossier courant
    print("Telechargement termine.")
else:
    print("Dataset deja present.")                   # cellule idempotente : relance sans risque

image_paths = sorted((DATA_ROOT / "PNGImages").glob("*.png"))  # toutes les photos, TRIEES
mask_paths = sorted((DATA_ROOT / "PedMasks").glob("*.png"))    # tous les masques, TRIES pareil
# * = joker ("n'importe quel nom"). Sans le *, glob(".png") chercherait un fichier
# appele litteralement ".png" -> liste vide (piege vecu !)

# comparaison chainee : (autant d'images que de masques) ET (au moins 1)
assert len(image_paths) == len(mask_paths) > 0
print(f"\n{len(image_paths)} couples image/masque disponibles")

# MISSION 1 — Lire une image, binariser un masque ✅

## 🧠 Les modes d'image PIL

| Mode | Un pixel = | Pour |
|---|---|---|
| `"RGB"` | 3 nombres (rouge, vert, bleu) | la **photo** (entrée du réseau) |
| `"L"` | 1 seul nombre 0-255 | le **masque** (0 = fond, 1, 2, 3… = personnes) |

Le masque a l'air « coloré » à l'affichage : c'est `cmap="nipy_spectral"` qui colorie
artificiellement les valeurs 0, 1, 2… pour l'œil. En mémoire : un entier par pixel.

## 🧠 La binarisation

On ne distingue pas la personne n°1 de la n°2 : ce TP fait juste « personne vs fond ».
`mask_array > 0` → `True` partout où il y a *quelqu'un*. `.astype(np.float32)` →
`1.0`/`0.0`, parce que la loss fera des **calculs continus** dessus (impossible avec des
booléens).

## 🧠 `.mean()` = un pourcentage

Sur un tableau qui ne contient que des 0 et des 1, la moyenne **est** la proportion de 1 —
donc le % de pixels « personne ».

In [ ]:
# --- MISSION 1 (resolue) ---

image = Image.open(image_paths[0]).convert("RGB")   # la 1re photo, forcee en couleur
mask_raw = Image.open(mask_paths[0]).convert("L")   # son masque, force en niveaux de gris

mask_array = np.array(mask_raw)                     # image PIL -> tableau NumPy 2D (H, W)
mask_binary = (mask_array > 0).astype(np.float32)   # binarise : tout sauf le fond -> 1.0

foreground_ratio = mask_binary.mean()               # moyenne de 0/1 = proportion de pixels "personne"

# --- Tests automatiques ---
assert image.mode == "RGB", "L'image doit etre en RGB"                # bien en couleur ?
assert mask_binary.ndim == 2, "Le masque doit avoir 2 dimensions"     # (H, W), pas de canal parasite
assert mask_binary.dtype == np.float32                                # le bon type pour la loss
assert set(np.unique(mask_binary)).issubset({0.0, 1.0})               # QUE des 0 et des 1
assert 0.0 < foreground_ratio < 1.0                                   # ni vide, ni tout plein

print(f"Forme image  : {np.array(image).shape}")     # (H, W, 3) : les canaux DERRIERE en NumPy
print(f"Forme masque : {mask_binary.shape}")         # (H, W)
print(f"Pixels personne : {100 * foreground_ratio:.1f} %")

# Affichage : photo / masque brut (couleurs artificielles) / masque binaire
fig, axes = plt.subplots(1, 3, figsize=(13, 4))
axes[0].imshow(image); axes[0].set_title("Image RGB")
axes[1].imshow(mask_raw, cmap="nipy_spectral"); axes[1].set_title("Masque brut (multi-classes)")
axes[2].imshow(mask_binary, cmap="gray"); axes[2].set_title("Masque binaire")
for ax in axes:
    ax.axis("off")                                   # pas de graduations sur des images
plt.show()

# 2 — Split train / validation / test

**La règle d'or, la même depuis S1** : trois jeux, trois rôles, jamais mélangés.

| Jeu | Part | Rôle |
|---|---|---|
| **train** | 70 % | ce sur quoi le modèle apprend |
| **validation** | 15 % | choisir le meilleur checkpoint **et** le seuil |
| **test** | 15 % | l'évaluation finale — on n'y touche qu'UNE fois, à la fin |

`random.Random(SEED)` crée un générateur **local** avec sa propre graine : le mélange est
reproductible sans toucher au générateur global.

In [ ]:
pairs = list(zip(image_paths, mask_paths))   # colle image[i] avec masque[i] -> couples
random.Random(SEED).shuffle(pairs)           # melange REPRODUCTIBLE (generateur local)

n_total = len(pairs)                         # nombre total de couples
n_train = int(0.70 * n_total)                # 70 % pour apprendre
n_val = int(0.15 * n_total)                  # 15 % pour valider

train_pairs = pairs[:n_train]                          # tranche 1 : le debut
val_pairs = pairs[n_train:n_train + n_val]             # tranche 2 : le milieu
test_pairs = pairs[n_train + n_val:]                   # tranche 3 : le reste (~15 %)

print(f"Train      : {len(train_pairs)}")
print(f"Validation : {len(val_pairs)}")
print(f"Test       : {len(test_pairs)}")

# isdisjoint : verifie qu'AUCUN exemple n'est dans deux jeux a la fois
assert set(train_pairs).isdisjoint(val_pairs)
assert set(train_pairs).isdisjoint(test_pairs)
assert set(val_pairs).isdisjoint(test_pairs)

# MISSION 2 — Le Dataset PyTorch ✅

## 🧠 Le chargement paresseux (*lazy loading*)

Différence majeure avec le `RegressionDataset` du S9_P4 : `__init__` ne stocke que des
**chemins**, pas les images. Chaque image n'est chargée **qu'au moment où le `DataLoader`
la demande** (`__getitem__`). Indispensable dès que le dataset ne tient plus en RAM.

## 🧠 BILINEAR vs NEAREST — le point le plus important

Redimensionner = **inventer** des pixels.

| Interpolation | Comment | Pour |
|---|---|---|
| `BILINEAR` | moyenne pondérée des voisins | la **photo** (couleurs douces) |
| `NEAREST` | recopie le pixel le plus proche | le **masque** (reste strictement 0/1) |

En `BILINEAR`, la frontière d'un masque 0/1 deviendrait `0.5` — ni fond, ni personne.
**Jamais** de moyenne sur un masque.

## 🧠 Le flip : MÊME tirage pour l'image ET le masque

Un seul `if` → soit les deux sont retournés, soit aucun. Sinon la personne serait à
droite sur la photo et son masque encore à gauche.

## 🧠 Les deux convertisseurs

- `TF.to_tensor(image)` fait **2 choses** : normalise 0-255 → 0-1 **et** réordonne
  `(H, W, 3)` → `(3, H, W)` (PyTorch veut les canaux DEVANT).
- Le masque passe par NumPy puis `torch.from_numpy(...)`, et `.unsqueeze(0)` ajoute le
  canal manquant : `(H, W)` → `(1, H, W)`.

In [ ]:
# --- MISSION 2 (resolue) ---

class PersonDataset(Dataset):                      # herite de Dataset = le contrat PyTorch
    def __init__(self, pairs, size=128, augment=False):
        self.pairs = list(pairs)                   # on stocke les CHEMINS, pas les images (lazy)
        self.size = size                           # taille cible du resize
        self.augment = augment                     # True = flips actives (train seulement)

    def __len__(self):
        return len(self.pairs)                     # "combien d'exemples ?" pour le DataLoader

    def __getitem__(self, index):                  # appele par le DataLoader, UN exemple a la fois
        image_path, mask_path = self.pairs[index]  # les 2 chemins de CET exemple

        image = Image.open(image_path).convert("RGB")   # charge la photo (couleur)
        mask = Image.open(mask_path).convert("L")       # charge le masque (niveaux de gris)

        # Redimensionnement : BILINEAR pour la photo...
        image = TF.resize(
            image, [self.size, self.size],
            interpolation=InterpolationMode.BILINEAR,
            antialias=True,                        # lissage propre au retrecissement
        )
        # ...NEAREST pour le masque (pas de moyenne -> reste strictement 0/1)
        mask = TF.resize(
            mask, [self.size, self.size],
            interpolation=InterpolationMode.NEAREST,
        )

        # Augmentation : 1 chance sur 2 de retourner horizontalement
        # UN SEUL if -> le MEME flip s'applique a l'image ET au masque
        if self.augment and random.random() < 0.5:
            image = TF.hflip(image)
            mask = TF.hflip(mask)

        image_tensor = TF.to_tensor(image)         # PIL -> tenseur (3,H,W), valeurs 0-1

        mask_np = (np.array(mask) > 0).astype(np.float32)   # binarise (comme Mission 1)
        mask_tensor = torch.from_numpy(mask_np).unsqueeze(0) # (H,W) -> (1,H,W) : canal explicite

        return image_tensor, mask_tensor           # LE tuple que le DataLoader empilera

# --- Tests automatiques ---
debug_dataset = PersonDataset(train_pairs[:3], size=IMG_SIZE, augment=False)

assert len(debug_dataset) == 3                      # __len__ repond juste ?
img_test, mask_test = debug_dataset[0]              # declenche __getitem__(0)

assert img_test.shape == (3, IMG_SIZE, IMG_SIZE)    # 3 canaux DEVANT (format PyTorch)
assert mask_test.shape == (1, IMG_SIZE, IMG_SIZE)   # le canal ajoute par unsqueeze(0)
assert img_test.dtype == torch.float32
assert mask_test.dtype == torch.float32
assert set(torch.unique(mask_test).tolist()).issubset({0.0, 1.0})  # NEAREST a preserve le binaire

print("MISSION 2 validee")
print(f"Image  : shape {img_test.shape}, dtype {img_test.dtype}")
print(f"Masque : shape {mask_test.shape}, dtype {mask_test.dtype}")

# Les DataLoaders

## Les règles (les mêmes que S9_P4)

| | `augment` | `shuffle` | Pourquoi |
|---|---|---|---|
| **train** | ✅ | ✅ | varier ce que le modèle voit |
| **val / test** | ❌ | ❌ | évaluer toujours pareil, reproductible |

## ⚠️ `num_workers=0` — le piège Windows vécu

`num_workers=2` lance 2 **processus** parallèles pour charger les images. Sous Linux : OK.
Sous **Windows + Jupyter** : ces processus doivent réimporter le notebook et plantent
(`DataLoader worker exited unexpectedly`). `0` = tout dans le processus principal.
Avec nos images, aucune différence de vitesse perceptible.

`pin_memory=True` : mémoire CPU « épinglée » → transferts vers le GPU plus rapides.

In [ ]:
train_dataset = PersonDataset(train_pairs, IMG_SIZE, augment=True)   # SEUL le train est augmente
val_dataset = PersonDataset(val_pairs, IMG_SIZE, augment=False)
test_dataset = PersonDataset(test_pairs, IMG_SIZE, augment=False)

train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True,   # melange chaque epoch
                          num_workers=0, pin_memory=True)    # 0 = pas de multiprocessing (Windows !)
val_loader = DataLoader(val_dataset, batch_size=BATCH_SIZE, shuffle=False,      # jamais melanger l'eval
                        num_workers=0, pin_memory=True)
test_loader = DataLoader(test_dataset, batch_size=BATCH_SIZE, shuffle=False,
                         num_workers=0, pin_memory=True)

# LE reflexe avant tout entrainement : tirer UN batch et verifier les formes
images_batch, masks_batch = next(iter(train_loader))
print(f"Batch images  : {images_batch.shape}")    # attendu : (8, 3, 128, 128)
print(f"Batch masques : {masks_batch.shape}")     # attendu : (8, 1, 128, 128)

In [ ]:
# Controle VISUEL : le masque #i doit epouser la personne de l'image #i
images_batch, masks_batch = next(iter(train_loader))   # un nouveau batch (train = melange)

fig, axes = plt.subplots(2, 4, figsize=(14, 7))        # 2 lignes (images / masques) x 4 colonnes
for i in range(4):
    # permute(1,2,0) : (3,H,W) -> (H,W,3), matplotlib veut les canaux DERRIERE
    # clamp(0,1) : force les valeurs dans [0,1] (filet de securite pour imshow)
    axes[0, i].imshow(images_batch[i].permute(1, 2, 0).clamp(0, 1))
    axes[0, i].set_title(f"Image #{i}")
    axes[0, i].axis("off")
    axes[1, i].imshow(masks_batch[i, 0], cmap="gray")  # [i, 0] : le canal 0 -> tableau 2D pur
    axes[1, i].set_title(f"Masque #{i}")
    axes[1, i].axis("off")
plt.tight_layout()
plt.show()

# 3 — Le U-Net

## 🧠 La convolution — LA nouveauté vs le cours

`nn.Linear` connecte chaque pixel à chaque neurone : il ignore la **position**. `Conv2d`
fait glisser un petit filtre 3×3 sur l'image :

- il exploite le fait que les pixels **voisins** sont liés (contours, textures)
- le **même** filtre sert partout → très peu de paramètres

`padding=1` ajoute une bordure de zéros pour que la taille H×W **ne change pas** —
indispensable pour recoller les skips plus tard.

## 🧠 BatchNorm2d

Normalise les sorties de la convolution à la volée (un `StandardScaler` intégré).
**Conséquence** : contrairement au MLP du cours, ce réseau se comporte **différemment**
en train et en eval → le `model.eval()` devient vraiment important.

## L'architecture

```
Input (3, 128, 128)
    ▼ enc1 ──────────── skip1 ──────────────┐
    ▼ pool (÷2)                             │
    ▼ enc2 ──────── skip2 ────────┐         │
    ▼ pool (÷2)                   │         │
  bottleneck  (le point le plus   │         │
    ▼ up2      compresse)         │         │
  concat ◄────────────────────────┘         │
    ▼ dec2                                  │
    ▼ up1                                   │
  concat ◄──────────────────────────────────┘
    ▼ dec1
    ▼ head → Output (1, 128, 128)   ← logits, PAS de sigmoid ici
```

**Les skip connections** : le pooling détruit le détail spatial (où est exactement le
contour ?). Les skips transportent ce détail de l'encoder directement au decoder —
sans elles, les masques seraient flous.

**Pourquoi pas de sigmoid à la fin** : la loss `BCEWithLogitsLoss` l'applique elle-même,
de façon numériquement plus stable. La sortie s'appelle des **logits**.

In [ ]:
class DoubleConv(nn.Module):
    """Bloc standard du U-Net : (Conv -> BN -> ReLU) x2"""

    def __init__(self, in_channels, out_channels):
        super().__init__()                          # initialise la mecanique nn.Module
        self.block = nn.Sequential(                 # enchaine les couches dans l'ordre
            nn.Conv2d(in_channels, out_channels, kernel_size=3, padding=1),  # filtre 3x3, taille conservee
            nn.BatchNorm2d(out_channels),           # normalise -> apprentissage stable
            nn.ReLU(inplace=True),                  # non-linearite (inplace = economie memoire)
            nn.Conv2d(out_channels, out_channels, kernel_size=3, padding=1), # 2e passe
            nn.BatchNorm2d(out_channels),
            nn.ReLU(inplace=True),
        )

    def forward(self, x):
        return self.block(x)                        # fait juste traverser le bloc

In [ ]:
# --- MISSION 3 (resolue) ---

class TinyUNet(nn.Module):
    def __init__(self, base=16):
        super().__init__()

        # --- Encoder : extraire puis retrecir, 2 fois ---
        self.enc1 = DoubleConv(3, base)             # 3 couleurs -> 16 caracteristiques
        self.pool1 = nn.MaxPool2d(2)                # divise H et W par 2 (garde le max local)

        self.enc2 = DoubleConv(base, base * 2)      # 16 -> 32
        self.pool2 = nn.MaxPool2d(2)                # encore /2

        # --- Bottleneck : le point le plus compresse (32x32, 64 canaux) ---
        self.bottleneck = DoubleConv(base * 2, base * 4)   # 32 -> 64

        # --- Decoder : regrandir puis recombiner avec le detail sauvegarde ---
        self.up2 = nn.ConvTranspose2d(base * 4, base * 2, kernel_size=2, stride=2)  # x2 (inverse du pool)
        self.dec2 = DoubleConv(base * 4, base * 2)  # entree = up (32) + skip2 (32) = 64

        self.up1 = nn.ConvTranspose2d(base * 2, base, kernel_size=2, stride=2)
        self.dec1 = DoubleConv(base * 2, base)      # entree = up (16) + skip1 (16) = 32

        # --- Head : 1 conv 1x1 -> 1 canal = la carte de logits ---
        self.head = nn.Conv2d(base, 1, kernel_size=1)

    def forward(self, x):
        # --- Encoder (on GARDE les sorties intermediaires : ce sont les skips) ---
        skip1 = self.enc1(x)                        # (16, 128, 128) - detail fin
        skip2 = self.enc2(self.pool1(skip1))        # (32, 64, 64)   - detail moyen
        x = self.bottleneck(self.pool2(skip2))      # (64, 32, 32)   - l'essentiel, tres compresse

        # --- Decoder niveau 2 ---
        x = self.up2(x)                             # regrandit : (32, 64, 64)
        x = torch.cat([x, skip2], dim=1)            # RECOLLE le detail : dim=1 = axe des canaux
        x = self.dec2(x)                            # recombine -> (32, 64, 64)

        # --- Decoder niveau 1 ---
        x = self.up1(x)                             # (16, 128, 128)
        x = torch.cat([x, skip1], dim=1)            # 2e skip connection
        x = self.dec1(x)                            # (16, 128, 128)

        return self.head(x)                         # (1, 128, 128) : des LOGITS (pas des probas)


# --- Tests automatiques ---
model = TinyUNet(BASE_CHANNELS).to(DEVICE)          # cree le reseau et l'envoie sur le device

dummy_input = torch.randn(2, 3, IMG_SIZE, IMG_SIZE, device=DEVICE)  # 2 fausses images (bruit)
dummy_output = model(dummy_input)                   # un forward pour verifier la PLOMBERIE

assert dummy_output.shape == (2, 1, IMG_SIZE, IMG_SIZE), \
    f"Shape sortie attendue (2, 1, {IMG_SIZE}, {IMG_SIZE}), obtenue {dummy_output.shape}"

# numel() = nombre d'elements d'un tenseur ; on somme sur tous les poids entrainables
n_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
print("MISSION 3 validee")
print(f"Parametres entrainables : {n_params:,}")
print(f"Input shape  : {dummy_input.shape}")
print(f"Output shape : {dummy_output.shape}")

# MISSION 4 — La loss ✅

## 🧠 Pourquoi deux losses ?

| Loss | Regarde | Force / faiblesse |
|---|---|---|
| **BCE** | chaque pixel séparément | précise, mais noyée si la personne est petite |
| **Dice** | le **recouvrement** des deux zones | robuste au déséquilibre fond/personne |

On prend moitié-moitié : le meilleur des deux mondes.

## 🧠 Soft Dice : pourquoi « soft » ?

Le vrai Dice compare des masques **binaires** (après seuillage). Mais le seuillage n'est
**pas différentiable** → pas de gradient → inutilisable comme loss. La *Soft* Dice
travaille directement sur les **probabilités** (sigmoid) : différentiable, donc
entraînable.

$$\text{dice} = \frac{2\sum(p \cdot t) + \varepsilon}{\sum p + \sum t + \varepsilon} \qquad \text{loss} = 1 - \text{dice}$$

L'`eps` évite 0/0 sur une image sans personne. `loss = 1 - dice` : dice parfait (1) →
loss 0.

## 🧠 Loss vs métrique — la distinction clé

- la **loss** (Soft Dice) : probabilités continues, sert à **entraîner**
- la **métrique** (Dice/IoU dans `binary_metrics`) : prédictions **seuillées**, sert à
  **évaluer** — c'est le chiffre que tu annonces

In [ ]:
# --- MISSION 4 (resolue) ---

BCE = nn.BCEWithLogitsLoss()               # BCE qui prend des LOGITS (sigmoid integre, + stable)


def soft_dice_loss(logits, targets, eps=1e-6):
    probabilities = torch.sigmoid(logits)  # logits -> probabilites 0-1 (differentiable)
    dims = (1, 2, 3)                       # sommer sur canaux+H+W, GARDER le batch (1 score/image)

    intersection = (probabilities * targets).sum(dim=dims)                 # ou les 2 sont d'accord
    denominator = probabilities.sum(dim=dims) + targets.sum(dim=dims)      # taille des 2 zones
    dice = (2 * intersection + eps) / (denominator + eps)                  # recouvrement dans [0,1]

    return 1 - dice.mean()                 # moyenne du batch, inversee (on MINIMISE une loss)


def combined_loss(logits, targets):
    """Moitie BCE (pixel par pixel) + moitie Dice (recouvrement global)."""
    return 0.5 * BCE(logits, targets) + 0.5 * soft_dice_loss(logits, targets)


@torch.no_grad()                           # metrique d'evaluation : pas besoin de gradient
def binary_metrics(logits, targets, threshold=0.5, eps=1e-6):
    """Dice et IoU par image du batch — sur des predictions SEUILLEES (0/1)."""
    probabilities = torch.sigmoid(logits)
    predictions = (probabilities >= threshold).float()   # ICI on binarise (pas differentiable, mais OK : on evalue)

    dims = (1, 2, 3)
    intersection = (predictions * targets).sum(dim=dims)
    pred_area = predictions.sum(dim=dims)
    target_area = targets.sum(dim=dims)
    union = pred_area + target_area - intersection       # |A u B| = |A| + |B| - |A n B|

    dice = (2 * intersection + eps) / (pred_area + target_area + eps)
    iou = (intersection + eps) / (union + eps)
    return dice, iou


# --- Tests automatiques ---
# logit +10 -> sigmoid = 0.99995 ; logit -10 -> 0.00005 : quasi 0/1 parfaits
perfect_targets = torch.tensor([[[[1., 0.], [0., 1.]]]], device=DEVICE)
perfect_logits = torch.tensor([[[[10., -10.], [-10., 10.]]]], device=DEVICE)
loss_perfect = soft_dice_loss(perfect_logits, perfect_targets).item()

bad_logits = -perfect_logits               # predictions inversees = le pire cas
loss_bad = soft_dice_loss(bad_logits, perfect_targets).item()

assert loss_perfect < 0.01                 # prediction parfaite -> loss ~0
assert loss_bad > 0.90                     # prediction inversee -> loss ~1

print("MISSION 4 validee")
print(f"Loss (prediction parfaite) : {loss_perfect:.4f}")
print(f"Loss (prediction inversee) : {loss_bad:.4f}")

# MISSION 5 — La boucle d'entraînement ✅

**Les 5 lignes sacrées** — les mêmes qu'au S9_P3, à jamais :

```
zero_grad → forward → loss → backward → step
```

Trois détails de pro dans cette cellule :

- `logits.detach()` avant les métriques : coupe le graphe d'autograd — on ne va pas
  dériver une métrique, inutile de garder l'historique
- `loss.item() * images.size(0)` : `.item()` extrait le scalaire ; on **pondère par la
  taille du batch** (le dernier batch peut être plus petit) puis on divise par le total →
  vraie moyenne
- `evaluate` porte `@torch.no_grad()` + `model.eval()` : pas de gradient, et BatchNorm
  passe en mode évaluation (ça compte, lui !)

In [ ]:
# --- MISSION 5 (resolue) ---

def train_one_epoch(model, loader, optimizer, device):
    model.train()                                   # mode entrainement (BatchNorm utilise le batch)

    total_loss = 0.0                                # accumulateur de loss ponderee
    all_dice = []                                   # les Dice de chaque image

    for images, masks in loader:                    # un batch a la fois
        images = images.to(device)                  # envoie les donnees sur le device...
        masks = masks.to(device)                    # ...modele ET donnees au MEME endroit

        optimizer.zero_grad()                       # 1. RAZ (sinon les gradients s'additionnent)
        logits = model(images)                      # 2. FORWARD : la prediction
        loss = combined_loss(logits, masks)         # 3. LOSS : l'erreur
        loss.backward()                             # 4. BACKWARD : autograd calcule les derivees
        optimizer.step()                            # 5. STEP : corrige les poids

        dice, _ = binary_metrics(logits.detach(), masks)   # detach : pas de gradient pour la metrique
        total_loss += loss.item() * images.size(0)  # somme ponderee par la taille du batch
        all_dice.extend(dice.cpu().tolist())        # rapatrie les scores en RAM

    return {
        "loss": total_loss / len(loader.dataset),   # vraie moyenne sur TOUT le dataset
        "dice": float(np.mean(all_dice)),
    }


@torch.no_grad()                                    # evaluation : aucun gradient nulle part
def evaluate(model, loader, device, threshold=0.5):
    """Evaluation sur un loader (val ou test). Retourne loss, dice, iou moyens."""
    model.eval()                                    # BatchNorm en mode evaluation !

    total_loss = 0.0
    all_dice = []
    all_iou = []

    for images, masks in loader:
        images = images.to(device)
        masks = masks.to(device)

        logits = model(images)                      # forward seulement (pas de backward ici)
        loss = combined_loss(logits, masks)
        dice, iou = binary_metrics(logits, masks, threshold)

        total_loss += loss.item() * images.size(0)
        all_dice.extend(dice.cpu().tolist())
        all_iou.extend(iou.cpu().tolist())

    return {
        "loss": total_loss / len(loader.dataset),
        "dice": float(np.mean(all_dice)),
        "iou": float(np.mean(all_iou)),
    }

# L'entraînement — avec le meilleur checkpoint

## 🧠 Pourquoi garder « le meilleur » et pas « le dernier » ?

La val peut se dégrader en fin d'entraînement (surapprentissage). On photographie donc
les poids **à chaque record de Dice validation** (`state_dict` cloné vers le CPU), et on
recharge cette photo à la fin. Le modèle final = le meilleur vu, pas le dernier.

In [ ]:
model = TinyUNet(BASE_CHANNELS).to(DEVICE)                  # reseau NEUF (poids reinitialises)
optimizer = torch.optim.Adam(model.parameters(), lr=LEARNING_RATE)   # Adam : le pas s'adapte tout seul

history = {"train_loss": [], "train_dice": [], "val_loss": [], "val_dice": []}   # pour les courbes
best_val_dice = -1.0                                        # le record a battre
best_state = None                                           # la "photo" des meilleurs poids

for epoch in range(1, EPOCHS + 1):
    train_stats = train_one_epoch(model, train_loader, optimizer, DEVICE)   # 1 passage complet
    val_stats = evaluate(model, val_loader, DEVICE)                          # controle sur la val

    history["train_loss"].append(train_stats["loss"])       # on note tout pour tracer ensuite
    history["train_dice"].append(train_stats["dice"])
    history["val_loss"].append(val_stats["loss"])
    history["val_dice"].append(val_stats["dice"])

    if val_stats["dice"] > best_val_dice:                   # nouveau record ?
        best_val_dice = val_stats["dice"]
        # state_dict = tous les poids ; on les CLONE vers le CPU (photo independante)
        best_state = {k: v.detach().cpu().clone() for k, v in model.state_dict().items()}
        marker = "  <- meilleur"
    else:
        marker = ""

    print(f"Epoch {epoch:02d}/{EPOCHS} | "
          f"train loss={train_stats['loss']:.3f}  dice={train_stats['dice']:.3f} | "
          f"val loss={val_stats['loss']:.3f}  dice={val_stats['dice']:.3f}{marker}")

assert best_state is not None
model.load_state_dict(best_state)                           # recharge la MEILLEURE photo
print(f"\nMeilleur Dice de validation : {best_val_dice:.4f}")

In [ ]:
# Les courbes : LE diagnostic d'un entrainement
epochs_range = range(1, EPOCHS + 1)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

axes[0].plot(epochs_range, history["train_loss"], "o-", label="Train")
axes[0].plot(epochs_range, history["val_loss"], "s-", label="Validation")
axes[0].set_xlabel("Epoch"); axes[0].set_ylabel("Loss")
axes[0].set_title("Loss"); axes[0].legend(); axes[0].grid(alpha=0.3)

axes[1].plot(epochs_range, history["train_dice"], "o-", label="Train")
axes[1].plot(epochs_range, history["val_dice"], "s-", label="Validation")
axes[1].set_xlabel("Epoch"); axes[1].set_ylabel("Dice")
axes[1].set_title("Dice score"); axes[1].legend(); axes[1].grid(alpha=0.3)
axes[1].set_ylim(0, 1)                                      # le Dice vit dans [0,1]

plt.tight_layout()
plt.show()

# Lecture : les 2 courbes descendent ensemble = ca apprend.
# train continue de monter mais val stagne/redescend = surapprentissage -> le
# meilleur checkpoint (garde plus haut) nous protege deja.

# MISSION 6 — Le seuil optimal ✅

## 🧠 Le seuil est un hyperparamètre comme un autre

`sigmoid` sort une probabilité par pixel ; il faut trancher « à partir de combien c'est
une personne ? ». `0.5` est le défaut, **pas forcément l'optimal** : si le modèle est
timide (probabilités basses), un seuil plus bas récupère des pixels corrects.

## ⚠️ On le cherche sur la VALIDATION, jamais sur le test

Chercher le meilleur seuil sur le test = régler un bouton en regardant la réponse =
**tricher**. Le test ne sert qu'une fois, tout à la fin, avec le seuil déjà décidé.
(Même logique que le `fit` du scaler sur le train seulement.)

In [ ]:
# --- MISSION 6 (resolue) ---

thresholds = [0.30, 0.40, 0.50, 0.60, 0.70]          # les candidats testes
validation_scores = []                                # leur Dice sur la VALIDATION

for threshold in thresholds:
    stats = evaluate(model, val_loader, DEVICE, threshold=threshold)   # meme modele, seuil different
    validation_scores.append(stats["dice"])
    print(f"Seuil {threshold:.2f} -> Dice validation = {stats['dice']:.4f}")

best_index = int(np.argmax(validation_scores))        # argmax = l'INDICE du meilleur score
best_threshold = thresholds[best_index]               # le seuil correspondant

assert best_threshold in thresholds
print(f"\nMeilleur seuil : {best_threshold:.2f}")

In [ ]:
# L'evaluation FINALE : le test set, une seule fois, avec le seuil decide sur la val
test_stats = evaluate(model, test_loader, DEVICE, threshold=best_threshold)

print("=== Resultats finaux sur le test set ===")
print(f"Loss  : {test_stats['loss']:.4f}")
print(f"Dice  : {test_stats['dice']:.4f}")            # LE chiffre a annoncer dans les slides
print(f"IoU   : {test_stats['iou']:.4f}")             # toujours < Dice (mathematiquement)

# Visualiser les prédictions

4 panneaux par image : photo / vérité / **probabilité** (avant seuil — le « doute » du
modèle en niveaux de gris) / **prédiction** (après seuil). C'est en regardant les erreurs
qu'on trouve les pistes d'amélioration.

In [ ]:
@torch.no_grad()                                     # visualisation : pas de gradient
def show_predictions(model, loader, threshold, n=4):
    model.eval()                                     # BatchNorm en mode eval
    images, masks = next(iter(loader))               # un batch du loader demande

    logits = model(images.to(DEVICE))                # forward
    probabilities = torch.sigmoid(logits).cpu()      # logits -> probas, rapatriees en RAM
    predictions = (probabilities >= threshold).float()   # seuillage -> masque 0/1

    n = min(n, len(images))                          # au cas ou le batch est petit
    fig, axes = plt.subplots(n, 4, figsize=(14, 3.5 * n))
    if n == 1:
        axes = axes.reshape(1, -1)                   # garde une grille 2D meme avec 1 ligne

    for i in range(n):
        axes[i, 0].imshow(images[i].permute(1, 2, 0).clamp(0, 1))     # la photo
        axes[i, 1].imshow(masks[i, 0], cmap="gray")                   # la verite terrain
        axes[i, 2].imshow(probabilities[i, 0], cmap="gray", vmin=0, vmax=1)  # le "doute"
        axes[i, 3].imshow(predictions[i, 0], cmap="gray")             # la decision finale
        for j in range(4):
            axes[i, j].axis("off")

    axes[0, 0].set_title("Image")
    axes[0, 1].set_title("Verite terrain")
    axes[0, 2].set_title("Probabilite")
    axes[0, 3].set_title("Prediction (seuillee)")
    plt.tight_layout()
    plt.show()


show_predictions(model, test_loader, best_threshold, n=4)

# MISSION 7 — Ta propre photo ✅

Le pipeline d'inférence sur UNE image, de bout en bout. Trois manipulations de forme à
bien voir :

- `.unsqueeze(0)` **avant** le modèle : `(3,H,W)` → `(1,3,H,W)` — le réseau attend
  toujours un batch, même d'une seule image
- `.squeeze()` **après** : retire toutes les dimensions de taille 1 → `(H,W)` affichable
- `.cpu().numpy()` : le GPU → la RAM → NumPy (matplotlib ne lit pas les tenseurs GPU)

In [ ]:
# --- MISSION 7 (resolue) ---

PHOTOS_DIR = Path("../photos_test")                   # le dossier ou deposer ta photo
PHOTOS_DIR.mkdir(exist_ok=True)                      # le cree s'il n'existe pas (idempotent)

photos = list(PHOTOS_DIR.glob("*.jpg")) + list(PHOTOS_DIR.glob("*.png"))   # jpg ET png
if not photos:
    raise FileNotFoundError(
        f"Aucune image trouvee dans {PHOTOS_DIR}. "
        f"Ajoutes-y une .jpg ou .png contenant une personne."
    )

photo_path = photos[0]                               # la premiere trouvee
print(f"Image testee : {photo_path.name}")

# --- Pretraitement : EXACTEMENT le meme que le Dataset (resize BILINEAR + to_tensor) ---
new_image = Image.open(photo_path).convert("RGB")
resized = TF.resize(new_image, [IMG_SIZE, IMG_SIZE],
                    interpolation=InterpolationMode.BILINEAR, antialias=True)
input_tensor = TF.to_tensor(resized).unsqueeze(0).to(DEVICE)   # (3,H,W) -> (1,3,H,W) -> device

# --- Inference : les 2 reflexes obligatoires ---
model.eval()                                         # BatchNorm en mode evaluation
with torch.no_grad():                                # pas de gradient en prediction
    logits = model(input_tensor)

probability = torch.sigmoid(logits).squeeze().cpu().numpy()    # (1,1,H,W) -> (H,W), en NumPy
prediction = (probability >= best_threshold).astype(np.float32) # seuillage avec le seuil APPRIS

# --- Tests automatiques ---
assert probability.shape == (IMG_SIZE, IMG_SIZE)
assert prediction.shape == (IMG_SIZE, IMG_SIZE)

fig, axes = plt.subplots(1, 3, figsize=(14, 5))
axes[0].imshow(new_image); axes[0].set_title("Photo originale"); axes[0].axis("off")
axes[1].imshow(probability, cmap="gray", vmin=0, vmax=1)
axes[1].set_title("Probabilite predite"); axes[1].axis("off")
axes[2].imshow(prediction, cmap="gray")
axes[2].set_title(f"Masque (seuil {best_threshold:.2f})"); axes[2].axis("off")
plt.tight_layout()
plt.show()

print("\nMISSION 7 validee")

# Bonus — le fond flou façon Zoom

L'idée : le masque prédit fait 128×128, mais ta photo est plus grande. On **re-agrandit
le masque** à la taille originale (en `NEAREST`, toujours !), puis :

- `~prediction_full` : le tilde inverse un tableau de booléens → **les pixels du fond**
- fond vert : on les peint en `[0, 255, 0]`
- fond flou : on les remplace par la version floutée (`GaussianBlur`)

⚠️ OpenCV travaille en **BGR** (l'ordre inverse de RGB) — d'où les deux `cvtColor`.

In [ ]:
import cv2                                # OpenCV : pip install opencv-python

# Le masque 128x128 -> retour a la taille ORIGINALE de la photo
original_size = new_image.size            # PIL donne (largeur, hauteur)
prediction_pil = Image.fromarray((prediction * 255).astype(np.uint8))   # 0/1 -> 0/255 en image
prediction_full = np.array(prediction_pil.resize(original_size, Image.NEAREST)) > 127  # booleen (H, W)

image_np = np.array(new_image)            # la photo en tableau NumPy

# --- Version fond vert ---
result_green = image_np.copy()            # copie (on ne touche pas a l'original)
result_green[~prediction_full] = [0, 255, 0]   # ~ = inversion booleenne -> LE FOND, peint en vert

# --- Version fond flou (style Zoom) ---
image_bgr = cv2.cvtColor(image_np, cv2.COLOR_RGB2BGR)   # OpenCV pense en BGR
blurred = cv2.GaussianBlur(image_bgr, (45, 45), 0)      # floute TOUTE l'image (noyau 45x45)
result_blur = image_bgr.copy()
result_blur[~prediction_full] = blurred[~prediction_full]   # ne garde le flou QUE sur le fond
result_blur = cv2.cvtColor(result_blur, cv2.COLOR_BGR2RGB)  # retour en RGB pour matplotlib

fig, axes = plt.subplots(1, 3, figsize=(14, 5))
axes[0].imshow(image_np); axes[0].set_title("Original"); axes[0].axis("off")
axes[1].imshow(result_green); axes[1].set_title("Fond vert"); axes[1].axis("off")
axes[2].imshow(result_blur); axes[2].set_title("Fond flou (style Zoom)"); axes[2].axis("off")
plt.tight_layout()
plt.show()

---

# 🎓 Bilan du projet v1

### Résultat de référence

Sur les données d'origine (170 images, 128px, 5 epochs) : **Dice test ≈ 0,64, IoU ≈ 0,49**.
*(Avec le dataset Kaggle de 2667 images dans le dossier, les scores montent.)*

### 🔴 Les 7 bugs vécus pendant CE projet — le vrai cours

| # | Bug | Symptôme | Leçon |
|---|---|---|---|
| 1 | `mask_array = np.array` sans `(...)` | `'>' not supported between builtin_function...` | `f` = la fonction, `f()` = son résultat |
| 2 | le `raise NotImplementedError` pas retiré | plante malgré le TODO rempli | Python lit ligne par ligne — la ligne suivante s'exécute quand même |
| 3 | `glob(".jpg")` sans `*` (+ mauvaise extension) | listes vides → `IndexError` | `*` = joker obligatoire ; vérifier la vraie extension |
| 4 | zip imbriqué `PennFudanPed/PennFudanPed/` | `glob` ne trouve rien | `rglob` cherche partout ; toujours vérifier ce qu'un zip a créé |
| 5 | `num_workers=2` sous Windows/Jupyter | `DataLoader worker exited unexpectedly` | `num_workers=0` sous Windows |
| 6 | kernel `python3` au lieu du venv | `No module named 'torchvision'` | vérifier QUEL Python exécute le notebook |
| 7 | MIOpen `type_traits` introuvable (ROCm) | `miopenStatusUnknownError` sur BatchNorm | ROCm compile à la volée → il lui faut un compilateur C++ (fix : headers MinGW dans `kernel.json`) |

**Le point commun des bugs 1-4 :** aucun ne venait du deep learning. Le métier, c'est
80 % de plomberie et 20 % de modèle.

### Les suites

- **v2** (`mini_projet_segmentation_v2_kaggle.ipynb`) : le même TP sur le dataset Kaggle
  Supervisely (2667 images), via `kagglehub`, pensé Google Colab
- **v3** (`mini_projet_segmentation_v3_pro.ipynb`) : la version **performance** —
  encoder ResNet18 pré-entraîné, 256px, augmentation riche, AdamW + cosine LR